# 06 — Layer A FAISS 全体再ビルド

**目的**: macOS CPU 環境では e5-large の推論メモリが不足するため、  
Colab の GPU (T4/A100) を使って Layer A FAISS インデックスを完全再ビルドする。

## 実行前チェックリスト
- [ ] ランタイムタイプが **GPU (T4 以上)** になっていることを確認
  - メニュー → ランタイム → ランタイムのタイプを変更 → T4 GPU

## ビルド内容
| 項目 | 値 |
|------|----|  
| モデル | `intfloat/multilingual-e5-large` |
| ベクトル次元 | 1024 |
| 対象レコード | 2,231 件（既存 2,040 + regulation 191） |
| インデックス型 | `IndexFlatIP`（内積・正規化済み → コサイン類似度） |
| 出力ファイル | `layer_a.index` + `layer_a_meta.json` |

---
## ステップ 0: GPU 確認

In [ ]:
import subprocess, sys

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print(f'✅ GPU 検出: {result.stdout.strip()}')
    HAS_GPU = True
else:
    print('⚠️  GPU が検出されません。CPU で続行します（エンコードが遅くなります）')
    HAS_GPU = False

print(f'Python: {sys.version}')

---
## ステップ 1: 依存関係インストール

`faiss-gpu` は CUDA バージョンの組み合わせによりインストールできない場合があります。  
その場合は自動的に `faiss-cpu` にフォールバックします。

In [ ]:
# sentence-transformers（エンコーダー）
print('sentence-transformers をインストール中...')
!pip install -q sentence-transformers numpy
print('✅ sentence-transformers インストール完了')

In [ ]:
# faiss: gpu → cpu の順でフォールバック
USE_GPU_FAISS = False

if HAS_GPU:
    print('faiss-gpu のインストールを試みます...')
    ret = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', 'faiss-gpu'],
        capture_output=True, text=True
    )
    if ret.returncode == 0:
        try:
            import faiss as _f
            if _f.get_num_gpus() > 0:
                USE_GPU_FAISS = True
                print(f'✅ faiss-gpu インストール成功 (GPUs: {_f.get_num_gpus()})')
            else:
                print('⚠️  faiss-gpu はインストールされたが GPU が認識されません → CPU に切り替え')
        except Exception as e:
            print(f'⚠️  faiss-gpu import エラー: {e} → CPU に切り替え')
    else:
        print('⚠️  faiss-gpu のインストールに失敗 → faiss-cpu にフォールバック')
        if ret.stderr:
            print('   エラー:', ret.stderr[-300:])

if not USE_GPU_FAISS:
    print('faiss-cpu をインストール中...')
    !pip install -q faiss-cpu
    print('✅ faiss-cpu インストール完了')

print(f'\nFAISS GPU モード: {USE_GPU_FAISS}')

In [ ]:
import importlib, json, time
from datetime import datetime, timezone
from pathlib import Path
from collections import Counter

# faiss を再インポート（インストール後に必要）
import importlib.util
if 'faiss' in sys.modules:
    import importlib
    faiss = importlib.reload(sys.modules['faiss'])
else:
    import faiss

import numpy as np
from sentence_transformers import SentenceTransformer

print(f'faiss バージョン: {faiss.__version__}')
print(f'faiss GPU 数: {faiss.get_num_gpus()}')
print(f'numpy バージョン: {np.__version__}')

---
## ステップ 2: データファイルの取得

必要なファイルは `layer_a_meta.json` だけです（`embed_text` が全件含まれています）。  

**取得方法を 1 つ選択してください:**
- **方法 A (推奨)**: GitHub から clone
- **方法 B**: Google Drive からマウント
- **方法 C**: 手動アップロード

In [ ]:
# ══════════════════════════════════════════════════
#  設定: 使用する方法を 1 つだけ True にしてください
# ══════════════════════════════════════════════════

METHOD_GITHUB = True    # GitHub clone（推奨）
METHOD_DRIVE  = False   # Google Drive マウント
METHOD_UPLOAD = False   # 手動アップロード

# GitHub 設定（METHOD_GITHUB = True の場合）
GITHUB_REPO   = 'https://github.com/tsp0918/AI_TradeManagement.git'
GITHUB_BRANCH = 'branch_neurosymbolic'

# Drive 設定（METHOD_DRIVE = True の場合）
DRIVE_REPO_ROOT = '/content/drive/MyDrive/AI_TradeManagement'

# ──────────────────────────────────────────────────
# 出力設定
SAVE_TO_DRIVE    = False   # True にすると Drive にも保存する
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/AI_TradeManagement/data/staging'

# ──────────────────────────────────────────────────
# モデル / インデックス設定
MODEL_NAME = 'intfloat/multilingual-e5-large'
BATCH_SIZE  = 128   # T4 16GB → 128, OOM なら 64 に下げる
DIM         = 1024

WORK_DIR = Path('/content/work')
WORK_DIR.mkdir(parents=True, exist_ok=True)

assert sum([METHOD_GITHUB, METHOD_DRIVE, METHOD_UPLOAD]) == 1, \
    'METHOD_* のうち 1 つだけ True にしてください'
print('設定 OK')

In [ ]:
REPO_ROOT = None

if METHOD_GITHUB:
    clone_dir = Path('/content/repo')
    if clone_dir.exists():
        print('既存の clone を更新します...')
        !git -C /content/repo pull
    else:
        print(f'GitHub から clone: {GITHUB_BRANCH}')
        ret = subprocess.run(
            ['git', 'clone', '--branch', GITHUB_BRANCH, '--depth', '1',
             GITHUB_REPO, str(clone_dir)],
            capture_output=True, text=True
        )
        if ret.returncode != 0:
            print('ERROR:', ret.stderr)
            raise RuntimeError('git clone failed')
    REPO_ROOT = clone_dir
    print(f'✅ リポジトリ: {REPO_ROOT}')

elif METHOD_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = Path(DRIVE_REPO_ROOT)
    print(f'✅ Drive マウント: {REPO_ROOT}')

elif METHOD_UPLOAD:
    from google.colab import files
    print('layer_a_meta.json をアップロードしてください')
    uploaded = files.upload()
    for fname, content in uploaded.items():
        (WORK_DIR / 'layer_a_meta.json').write_bytes(content)
    print('✅ アップロード完了')

# META_PATH を決定
if REPO_ROOT:
    META_PATH = REPO_ROOT / 'data' / 'staging' / 'layer_a_meta.json'
else:
    META_PATH = WORK_DIR / 'layer_a_meta.json'

assert META_PATH.exists(), f'ファイルが見つかりません: {META_PATH}'
print(f'Meta ファイル: {META_PATH} ({META_PATH.stat().st_size // 1024} KB)')

In [ ]:
with META_PATH.open(encoding='utf-8') as f:
    meta = json.load(f)

records = meta['records']
print(f'総レコード数: {len(records)}')
print(f'モデル: {meta.get("model")}')
print()

ct = Counter(r.get('source_type', '?') for r in records)
print('source_type 別件数:')
for k, v in sorted(ct.items()):
    mark = ' ← 今回追加（pending）' if k == 'regulation' else ''
    print(f'  {k:15s}: {v:5d}{mark}')

pending = [r for r in records if r.get('_pending_embed')]
print(f'\n_pending_embed フラグ付き: {len(pending)} 件')

# embed_text が全件に存在するか確認
missing_embed = sum(1 for r in records if not r.get('embed_text', '').strip())
print(f'embed_text が空: {missing_embed} 件')
if missing_embed:
    print('  ⚠️  空の embed_text があります。エンコード時にスキップされます。')

---
## ステップ 3: モデルのロードとエンコード

In [ ]:
# GPU が使えれば cuda、なければ cpu
import torch
DEVICE = 'cuda' if (HAS_GPU and torch.cuda.is_available()) else 'cpu'
print(f'使用デバイス: {DEVICE}')

print(f'モデルをロード中: {MODEL_NAME} ...')
t0 = time.time()
model = SentenceTransformer(MODEL_NAME, device=DEVICE)
print(f'✅ ロード完了 ({time.time()-t0:.1f}s)')
print(f'   埋め込み次元: {model.get_sentence_embedding_dimension()}')

if DEVICE == 'cuda':
    !nvidia-smi --query-gpu=memory.used,memory.free --format=csv,noheader

In [ ]:
# 全 2,231 件をエンコード
embed_texts = [r.get('embed_text', '') for r in records]

empty_idx = [i for i, t in enumerate(embed_texts) if not t.strip()]
if empty_idx:
    print(f'⚠️  embed_text が空のレコード: {len(empty_idx)} 件 → 空文字列で埋め込みます')
    for i in empty_idx:
        embed_texts[i] = 'passage: （テキストなし）'

print(f'エンコード開始: {len(embed_texts)} 件 / batch_size={BATCH_SIZE} / device={DEVICE}')
print('（T4 GPU で約 4〜6 分、CPU のみで約 30〜60 分かかります）')
t0 = time.time()

embeddings = model.encode(
    embed_texts,
    batch_size=BATCH_SIZE,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True,
)

elapsed = time.time() - t0
vectors = np.asarray(embeddings, dtype='float32')
print(f'\n✅ エンコード完了: {elapsed:.1f}s ({elapsed/60:.1f}min)')
print(f'   ベクトル形状: {vectors.shape}')
print(f'   ノルム確認（先頭3件）: {np.linalg.norm(vectors[:3], axis=1).tolist()}')

---
## ステップ 4: FAISS インデックス構築

In [ ]:
print(f'IndexFlatIP を構築 (dim={DIM}, USE_GPU_FAISS={USE_GPU_FAISS})')

if USE_GPU_FAISS and faiss.get_num_gpus() > 0:
    # GPU FAISS: 高速ビルド後に CPU インデックスへ転送
    print('  GPU インデックスを使用...')
    res = faiss.StandardGpuResources()
    gpu_index = faiss.GpuIndexFlatIP(res, DIM)
    gpu_index.add(vectors)
    cpu_index = faiss.index_gpu_to_cpu(gpu_index)
    del gpu_index
else:
    # CPU FAISS: faiss-gpu 不要、faiss-cpu で動作
    print('  CPU インデックスを使用...')
    cpu_index = faiss.IndexFlatIP(DIM)
    cpu_index.add(vectors)

print(f'✅ インデックス構築完了: ntotal={cpu_index.ntotal}')
assert cpu_index.ntotal == len(records), \
    f'件数不一致: index={cpu_index.ntotal}, records={len(records)}'

In [ ]:
# 動作確認: サンプルクエリで検索
def search_index(query: str, top_k: int = 3):
    q_vec = model.encode([f'query: {query}'], normalize_embeddings=True)
    D, I = cpu_index.search(np.asarray(q_vec, dtype='float32'), top_k)
    print(f'\nQuery: "{query}"')
    for rank, (score, idx) in enumerate(zip(D[0], I[0])):
        if idx < 0:
            continue
        rec   = records[idx]
        src   = rec.get('source_type', '?')
        title = (rec.get('title') or rec.get('label') or
                 rec.get('category') or rec.get('item_no') or '')[:50]
        node  = rec.get('node_id', '') or rec.get('item_no', '')
        print(f'  [{rank+1}] score={score:.4f} | {src:10s} | {node:12s} | {title}')

search_index('工作機械 位置決め精度 規制')
search_index('nuclear reactor uranium enrichment')
search_index('半導体製造装置 ECCN')
search_index('ミサイル ロケット 推進装置')
search_index('化学兵器 前駆体 輸出規制')

---
## ステップ 5: メタデータ更新と保存

In [ ]:
# faiss_id を再割り当て + _pending_embed フラグを削除
for i, rec in enumerate(records):
    rec['faiss_id'] = i
    rec.pop('_pending_embed', None)

src_breakdown = dict(Counter(r.get('source_type', '?') for r in records))

new_meta = {
    'total':            cpu_index.ntotal,
    'dim':              DIM,
    'model':            MODEL_NAME,
    'built_at':         datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
    'source_breakdown': src_breakdown,
    'rebuild_notes': (
        f'Full rebuild via Colab (device={DEVICE}, faiss_gpu={USE_GPU_FAISS}). '
        f'Added 191 regulation nodes from control_nodes.json (2026-03-21). '
        f'Model: {MODEL_NAME}.'
    ),
    'records': records,
}

print('更新後メタデータ:')
for k, v in new_meta.items():
    if k != 'records':
        print(f'  {k}: {v}')
print(f'  records: {len(records)} 件')

In [ ]:
OUT_INDEX = WORK_DIR / 'layer_a.index'
OUT_META  = WORK_DIR / 'layer_a_meta.json'

faiss.write_index(cpu_index, str(OUT_INDEX))
with OUT_META.open('w', encoding='utf-8') as f:
    json.dump(new_meta, f, ensure_ascii=False)

print('✅ 保存完了')
print(f'   layer_a.index    : {OUT_INDEX.stat().st_size / 1024 / 1024:.1f} MB')
print(f'   layer_a_meta.json: {OUT_META.stat().st_size / 1024:.0f} KB')

# Drive にも保存
if SAVE_TO_DRIVE:
    import shutil
    drive_out = Path(DRIVE_OUTPUT_DIR)
    drive_out.mkdir(parents=True, exist_ok=True)
    shutil.copy(OUT_INDEX, drive_out / 'layer_a.index')
    shutil.copy(OUT_META,  drive_out / 'layer_a_meta.json')
    print(f'✅ Drive にも保存: {drive_out}')

---
## ステップ 6: ダウンロード

生成した 2 ファイルをローカルにダウンロードし `data/staging/` に配置します。

In [ ]:
from google.colab import files

print('--- layer_a.index をダウンロード ---')
files.download(str(OUT_INDEX))

print('--- layer_a_meta.json をダウンロード ---')
files.download(str(OUT_META))

print()
print('✅ ダウンロード完了')
print()
print('【次のステップ（ローカル）】')
print('  1. ダウンロードしたファイルを data/staging/ に上書き配置')
print('  2. platform-core を再起動')
print('  3. 動作確認:')
print('     curl "http://localhost:8000/api/faiss/search/layer-a?q=工作機械規制&top_k=3"')

---
## 付録: トラブルシューティング

### OOM (RuntimeError: CUDA out of memory)
```python
BATCH_SIZE = 64   # 128 → 64 に下げる
BATCH_SIZE = 32   # それでも OOM なら
```

### `faiss` が import できない
セルを上から順番に実行してください（インストールより先に import しているのが原因です）。

### `git clone` が失敗する（Private リポジトリの場合）
```python
GITHUB_REPO = 'https://<YOUR_TOKEN>@github.com/tsp0918/AI_TradeManagement.git'
```

### ローカルで確認
```python
import faiss, json
idx  = faiss.read_index('data/staging/layer_a.index')
meta = json.load(open('data/staging/layer_a_meta.json'))
print('ntotal:', idx.ntotal)           # → 2231
print('breakdown:', meta['source_breakdown'])
# → {'fefta_law': 944, 'fefta_parameter': 406,
#    'fefta_tsutatsu': 53, 'eccn': 637, 'regulation': 191}
```